# 02 · Baseline: HOG + logistic regression (Person 1)
Hand-crafted HOG features on the 64x64 crops from notebook 01, a linear classifier, and a threshold chosen on validation.

Protocol: hyperparameters by **val ROC-AUC** → threshold by **val balanced accuracy (Youden J)** → model and threshold **frozen** to `models/` → test split scored **once**.

Set `RUN_TRAINING = True` to retrain from scratch (≈3.5 min on 2 CPUs). Otherwise the notebook loads the frozen model and checks it reproduces the saved numbers.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from IPython.display import Image, display
pd.set_option("display.width", 140)

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    from src.models import train_baseline
    train_baseline.main([])
S = json.loads(Path("results/baseline/baseline_summary.json").read_text())

## 1 · Hyperparameter search (validation only)

In [ ]:
for t in ["genki_smile", "fer_expression"]:
    print(t); display(pd.read_csv(f"results/baseline/{t}_grid.csv").head(6))

In [ ]:
display(Image("results/baseline/hog_visualization.png"))

## 2 · Frozen configuration

In [ ]:
for t in ["genki_smile", "fer_expression"]:
    meta = json.loads(Path(f"models/baseline_{t}.json").read_text())
    print(t, json.dumps(meta["config"]), "| frozen", meta["frozen_at"], "| split md5", meta["split_file_md5"],
          "| strict threshold", round(meta["alt_threshold_fpr5"], 4))

## 3 · Results

In [ ]:
cols = ["threshold","accuracy","balanced_accuracy","precision","recall","f1","false_alarm_rate","roc_auc","pr_auc"]
rows = []
for t in ["genki_smile", "fer_expression"]:
    for sp in ["val", "test"]:
        for op in ["default_threshold", "strict_threshold"]:
            rows.append({"task": t, "split": sp, "operating point": op.replace("_threshold",""),
                         **{c: S[t][sp][op][c] for c in cols}})
pd.DataFrame(rows).set_index(["task","split","operating point"]).round(3)

In [ ]:
for t in ["genki_smile", "fer_expression"]:
    display(Image(f"results/baseline/{t}_roc_pr.png")); display(Image(f"results/baseline/{t}_confusion.png"))

## 4 · Where GENKI errors come from

In [ ]:
t = S["genki_smile"]["test"]
display(pd.DataFrame({k: {"n": v["n"], "accuracy": v["accuracy"], "roc_auc": v["roc_auc"]} for k, v in t["by_face_method"].items()}).T)
if "by_abs_yaw_deg" in t:   # only when GENKI head-pose labels are available
    display(pd.DataFrame({k: {"n": v["n"], "accuracy": v["accuracy"], "roc_auc": v["roc_auc"], "false_alarm_rate": v["false_alarm_rate"]} for k, v in t["by_abs_yaw_deg"].items()}).T)
display(Image("results/baseline/genki_smile_errors.png"))

## 5 · FER: which expressions get flagged, and cross-dataset transfer

In [ ]:
display(pd.Series(S["fer_expression"]["test"]["by_fer_class_flag_rate"], name="flag rate at default threshold").round(3).to_frame())
display(pd.DataFrame({k: {c: v[c] for c in ["n","accuracy","balanced_accuracy","recall","false_alarm_rate","roc_auc"]} for k, v in S["cross_domain"].items()}).T)
display(Image("results/baseline/fer_expression_errors.png"))

## 6 · Reproducibility check: the frozen model reproduces saved val metrics

In [ ]:
from sklearn.metrics import roc_auc_score
from src.models.baseline import HOGBaseline
from src.models.train_baseline import load
for t, tag in [("genki_smile","genki4k"), ("fer_expression","fer2013")]:
    df, X = load(tag); m = (df["split"] == "val").values
    auc = roc_auc_score(df.loc[m, "game_label"], HOGBaseline.load(Path("models")/f"baseline_{t}").score(X[m]))
    print(t, "val AUC now", round(auc, 4), "| saved", S[t]["val"]["default_threshold"]["roc_auc"])
    assert abs(auc - S[t]["val"]["default_threshold"]["roc_auc"]) < 1e-3

## 7 · Webcam-frame entry point
`src/models/baseline_detector.py` wraps the frozen GENKI model for live frames, in both the README contract and the app adapter contract.

In [ ]:
import cv2
from src.models.baseline_detector import predict, predict_expression
df = pd.read_csv("data/splits/genki4k.csv")
r = df[(df.split == "test") & df.keep & df.face_method.str.startswith("haar")].iloc[0]
img = cv2.imread("data/raw/genki4k/" + r.rel_path)
print("label:", r.raw_label); print(predict(img)); print(predict_expression(img, 0.0))